# 멤버 1번 소비 패턴 및 이상치 비교 분석

이 노트북은 과거 데이터(1~3월)와 오늘 데이터(4/1)를 비교하여, IQR 클리핑 기법을 통한 안정적 지표 산출 및 특이 행동 탐지를 수행합니다.

In [33]:
import pandas as pd
import numpy as np
from datetime import datetime

# 1. 데이터 로드
past_data_path = '../../../data/raw/csv/consumption_v1.csv'
today_data_path = './data_input_month.csv'

df_past = pd.read_csv(past_data_path)
df_today_full = pd.read_csv(today_data_path)

# 과거 데이터에서 멤버 1번만 추출
df_m1 = df_past[df_past['멤버 id'] == 1].copy()
df_m1['사용 시간'] = pd.to_datetime(df_m1['사용 시간'])
df_m1['date'] = df_m1['사용 시간'].dt.date

# 오늘 데이터 (2024-04-01) 추출
df_today_full['사용 시간'] = pd.to_datetime(df_today_full['사용 시간'])
df_today = df_today_full[df_today_full['사용 시간'].dt.strftime('%Y-%m-%d') == '2024-04-01'].copy()

print(f"과거 데이터 로드 완료: {len(df_m1)}건")
print(f"오늘(4/1) 데이터 로드 완료: {len(df_today)}건")

과거 데이터 로드 완료: 465건
오늘(4/1) 데이터 로드 완료: 9건


## 2. 과거 데이터 이상치 처리 (IQR 클리핑)
지나치게 높은 고액 소비(이상치)가 지표를 왜곡하지 않도록 클리핑을 적용합니다.

In [34]:
# IQR 기반 클리핑 계산
Q1 = df_m1['사용 금액'].quantile(0.25)
Q3 = df_m1['사용 금액'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = max(0, Q1 - 1.5 * IQR)

# 클리핑 적용 컬럼 생성
df_m1['사용 금액_clipped'] = df_m1['사용 금액'].clip(lower=lower_bound, upper=upper_bound)

# 일별 합계 데이터 생성
daily_orig = df_m1.groupby('date')['사용 금액'].sum()
daily_clipped = df_m1.groupby('date')['사용 금액_clipped'].sum()

# 일일 지표 비교 데이터프레임
comparison = pd.DataFrame({
    '구분': ['일일 원본 (Original Daily)', '일일 클리핑 후 (Clipped Daily)'],
    '평균 (Mean)': [daily_orig.mean(), daily_clipped.mean()],
    '중앙값 (Median)': [daily_orig.median(), daily_clipped.median()],
    '표준편차 (Std)': [daily_orig.std(), daily_clipped.std()],
    '최댓값 (Max)': [daily_orig.max(), daily_clipped.max()]
})

print("--- 과거 데이터 지표 비교 ---")
display(comparison)

--- 과거 데이터 지표 비교 ---


,구분,평균 (Mean),중앙값 (Median),표준편차 (Std),최댓값 (Max)
0,일일 원본 (Original Daily),104423.901099,52742.0,339117.355263,2923962.0
1,일일 클리핑 후 (Clipped Daily),51014.054945,49815.5,16051.611825,103734.5


## 3. 오늘(4/1) 데이터 분석 및 과거와 비교
오늘의 지출이 과거의 안정적인 소비 패턴(Clipped)과 어떻게 다른지 분석합니다.

In [35]:
# 오늘(4/1)의 총 지출액
today_total = df_today['사용 금액'].sum()
past_stable_avg = daily_clipped.mean()

print(f"오늘(4/1) 총 지출액: {today_total:,.0f}원")
print(f"과거 일일 안정적 평균(Clipped): {past_stable_avg:,.0f}원")
print(f"평균 대비 지출율: {(today_total / past_stable_avg) * 100:.2f}%")

# 카테고리별 비중 분석
today_cat_ratio = df_today.groupby('업종 카테고리')['사용 금액'].sum() / today_total * 100
print("\n--- 오늘 카테고리별 지출 비중 (%) ---")
print(today_cat_ratio)

오늘(4/1) 총 지출액: 133,044원
과거 일일 안정적 평균(Clipped): 51,014원
평균 대비 지출율: 260.80%

--- 오늘 카테고리별 지출 비중 (%) ---
업종 카테고리
교통     9.688524
생활    72.833048
식비     7.385527
의료    10.092902
Name: 사용 금액, dtype: float64


## 4. [분석 결과 요약]

### [1] 클리핑 데이터 → 안정적 지표
- **평균 대비**: 오늘의 지출이 과거의 '평균적인' 날들에 비해 얼마나 높은지 확인하여 과소비 여부를 판단합니다.
- **카테고리 비율**: 오늘 유독 '식비'나 '쇼핑' 비중이 과거 평균보다 높은지 체크합니다.

### [2] 원본 데이터 → 행동 탐지 (이상치 분석)
- **과거 특이 이벤트**: 과거 최댓값({daily_orig.max():,.0f}원)은 주로 의료(종합병원)나 쇼핑(애플스토어) 같은 특이 상황에서 발생했습니다.
- **오늘의 특이점**: 오늘 데이터 중 건당 금액이 상한선({upper_bound:,.0f}원)을 넘는 건이 있는지 확인합니다.

In [37]:
# 오늘 데이터 중 과거 기준 이상치에 해당하는 건 탐지
today_outliers = df_today[df_today['사용 금액'] > upper_bound]
if not today_outliers.empty:
    print("오늘 발견된 고액 지출(이상치) 건:")
    display(today_outliers[['결제 내역', '사용 금액', '업종 카테고리']])
else:
    print("오늘은 과거 기준을 크게 벗어나는 고액 지출이 없습니다.")

오늘 발견된 고액 지출(이상치) 건:


,결제 내역,사용 금액,업종 카테고리
2,SKT통신비,65000,생활
